
# DAT & XYZ (Loupe TEM)
This example demonstrates conversion of data from Loupe Geophysics' time domain electromagnetic (TEM) backpack system.

The raw data is in .dat format as exported by the Loupe. The inverted resistivity models come as a set of syn/dat/inv .XYZ files exported by Aarhus Workbench.

Source Reference: Minsley et al., in preparation, LoupeEM data and resistivity models from the Kankakee River groundwater-surface water interaction site, U.S. Geological Survey data release.


In [ ]:
import matplotlib.pyplot as plt
from os.path import join
import numpy as np
import gspy
import warnings
warnings.filterwarnings('ignore')

## Initialize the Survey



In [ ]:
# Path to example files
data_path = '../data_files/loupe'

# Survey metadata file
metadata = join(data_path, "data", "LoupeEM_survey_md.yml")

Establish the Survey



In [ ]:
survey = gspy.Survey.from_dict(metadata)

.. literalinclude:: /../../examples/data_files/loupe/data/LoupeEM_survey_md.yml
   :language: yaml
   :linenos:
   :caption: Survey YAML file




## Create a 'data' branch and attach data leaves



In [ ]:
data_container = survey.gs.add_container('data', **dict(content='data container'))

Attach raw data with the Loupe EM system



In [ ]:
data = join(data_path, 'data', 'Kankakee.dat')
metadata = join(data_path, 'data', 'Loupe_data_metadata.yml')

In [ ]:
raw_data = data_container.gs.add(key='raw_data', data=data, 
                                 metadata_file=metadata, file_type='loupe')

.. literalinclude:: /../../examples/data_files/loupe/data/Loupe_data_metadata.yml
   :language: yaml
   :linenos:
   :caption: Raw Data YAML file




The long names of the raw columns, including oddities like a leading "(m)"
or "(UNITS)", are transcribed from the survey's original .dat.desc file that
accompanies the Loupe export; gspy keeps contractor metadata as-is rather
than rewriting it. What the .desc file does not declare - units, missing
values, the gate-times dimension, and each EM column's system couplet -
comes from the 'variables' block of the metadata YAML, merged on top. Where
the two do describe the same field, the .desc wording is preserved. The
ASEG-GDF2 example shows the case where the YAML does overwrite the file's
own descriptions, field by field.



## Create a 'models' branch and attach data leaves



Make the branch



In [ ]:
model_container = survey.gs.add_container('models', **dict(content='models container'))

Import Workbench XYZ inversion results (syn/dat/inv)



GSPy automatically recognizes Workbench inversion files (syn/dat/inv), only
one of the three files need to be passed to provide the base filename



In [ ]:
d_data = join(data_path, 'model', 'Kankakee_MOD_dat.xyz')
d_supp = join(data_path, 'model', 'models.yml')

In [ ]:
md = model_container.gs.add(key='inversion', data=d_data, metadata_file=d_supp)

.. literalinclude:: /../../examples/data_files/loupe/model/models.yml
   :language: yaml
   :linenos:
   :caption: Inverted Models YAML file




## Save to NetCDF file



In [ ]:
d_out = join(data_path, 'Loupe.nc')
survey.gs.to_netcdf(d_out)

Reading back in the GS NetCDF file



In [ ]:
new_survey = gspy.open_datatree(d_out)['survey']

## View the Data Tree



In [ ]:
print(new_survey.gs.tree)
print(new_survey)

## Plotting



In [ ]:
# Line profiles of the antenna and coil heights along one traverse, subset by
# line so the title records which one. The axis labels drawn from the variable
# metadata are then overwritten with shorter wording.
line = int(new_survey['data/raw_data']['line'].values[0])
subset = new_survey['data/raw_data'].gs.subset('line', line)

plt.figure()
subset.gs.plot('gps_height', x='distance', label='GPS height')
subset.gs.plot('tx_height', x='distance', label='TX height')
subset.gs.plot('rx_height', x='distance', label='RX height')
plt.xlabel('Distance along line (m)')
plt.ylabel('Height (m)')
plt.title(f"Line {line}")
plt.tight_layout()
plt.legend()

A map view of the traverses colored by height, using GSPy's plotter.



In [ ]:
plt.figure()
new_survey['data/raw_data'].gs.scatter(hue='z', cmap='viridis')

plt.show()